In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler


In [2]:
raw = np.load("/content/PEMS08.npz")
data = raw["data"].astype(np.float32)

# Use traffic flow channel only
data = data[:, :, 0]      # (time, sensors)

SENSORS = data.shape[1]
print("Shape:", data.shape)


Shape: (17856, 170)


In [3]:
scaler = MinMaxScaler()
data = scaler.fit_transform(data)


In [4]:
SEQ_LEN = 96
PRED_LEN = 12

def create_sequences(data):
    X, Y = [], []
    for i in range(len(data) - SEQ_LEN - PRED_LEN):
        X.append(data[i:i+SEQ_LEN])
        Y.append(data[i+SEQ_LEN:i+SEQ_LEN+PRED_LEN])
    return np.array(X), np.array(Y)

X, Y = create_sequences(data)
print(X.shape, Y.shape)


(17748, 96, 170) (17748, 12, 170)


In [5]:
class TrafficDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X).float()
        self.Y = torch.tensor(Y).float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

dataset = TrafficDataset(X, Y)


In [8]:
split = int(0.8 * len(dataset))

train_set = torch.utils.data.Subset(dataset, range(split))
val_set   = torch.utils.data.Subset(dataset, range(split, len(dataset)))

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=32)


In [9]:
class LSTM_LLM_PRO(nn.Module):
    def __init__(self, sensors, pred_len):
        super().__init__()

        hidden = 128

        self.input_proj = nn.Linear(sensors, hidden)

        self.lstm = nn.LSTM(
            input_size=hidden,
            hidden_size=hidden,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden,
            nhead=8,
            dim_feedforward=256,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=3
        )

        self.attn = nn.MultiheadAttention(hidden, 8, batch_first=True)

        self.fc = nn.Linear(hidden, sensors * pred_len)

        self.pred_len = pred_len
        self.sensors = sensors

    def forward(self, x):

        x = self.input_proj(x)

        x, _ = self.lstm(x)

        x = self.transformer(x)

        attn_out, _ = self.attn(x, x, x)
        x = x + attn_out

        out = self.fc(x[:, -1, :])

        return out.reshape(-1, self.pred_len, self.sensors)


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTM_LLM_PRO(SENSORS, PRED_LEN).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=3, factor=0.5
)

loss_fn = nn.HuberLoss()


In [11]:
best_val = 1e9
patience = 8
counter = 0

for epoch in range(60):

    model.train()
    train_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            val_loss += loss_fn(pred, y).item()

    val_loss /= len(val_loader)
    scheduler.step(val_loss)

    print(epoch, train_loss, val_loss)

    if val_loss < best_val:
        best_val = val_loss
        counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break


0 0.017303720436495234 0.0049630267470068224
1 0.004302109617358158 0.003955549512243083
2 0.003707096512903468 0.003470149009199058
3 0.0031956478886239164 0.0032007282915491576
4 0.0028934569663351386 0.00294797178483694
5 0.0026418776993392254 0.0027577270672925086
6 0.002458976276495771 0.002637384765097653
7 0.0023213325730139004 0.002604080447486627
8 0.0021919817333361387 0.0026591997834862218
9 0.0020668617237845923 0.0023584035277651907
10 0.001964403242701329 0.002246171204836804
11 0.001902343562137134 0.002230771273208849
12 0.001833547144784248 0.0022428583228952244
13 0.0017774333109709997 0.0021085411775332817
14 0.0017339362818454098 0.0021380518661677705
15 0.0016972076261588078 0.002174366501774555
16 0.0016422051176542064 0.0020446948243540014
17 0.0016000306313471483 0.002040178788750357
18 0.0015805180833217696 0.002118312774188313
19 0.001543834735744272 0.002032322584397604
20 0.0015152756602981605 0.0020111594949576267
21 0.0014797412071414794 0.0019847740105097

In [12]:
model.load_state_dict(torch.load("best_model.pt"))
model.eval()

preds, actual = [], []

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)
        p = model(x).cpu().numpy()
        preds.append(p)
        actual.append(y.numpy())

preds = np.concatenate(preds)
actual = np.concatenate(actual)

# Inverse scaling
preds_denorm = scaler.inverse_transform(
    preds.reshape(-1, preds.shape[-1])
).reshape(preds.shape)

actual_denorm = scaler.inverse_transform(
    actual.reshape(-1, actual.shape[-1])
).reshape(actual.shape)

mae  = np.mean(np.abs(preds_denorm - actual_denorm))
rmse = np.sqrt(np.mean((preds_denorm - actual_denorm)**2))

mask = actual_denorm > 10
mape = np.mean(
    np.abs((actual_denorm[mask] - preds_denorm[mask]) /
           actual_denorm[mask])
) * 100

print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)


MAE : 18.915453
RMSE: 29.564093
MAPE: 10.671251
